# Preprocessing diagnostics — W+/W-/Z/QCD

Sanity-check plots for the `boostedVtagger` preprocessing output, ahead of training.
Point `PARQUET_DIR` at a directory of per-dataset `.parquet` files produced by `run.py` (or their merge).

In [ ]:
import glob
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

hep.style.use("CMS")

PARQUET_DIR = "../../vjet_preprocess_20260705"  # point this at your output directory

CLASS_ORDER = ["Wplus", "Wminus", "Z", "QCD"]
CLASS_COLORS = {
    "Wplus": "#5790fc",
    "Wminus": "#e42536",
    "Z": "#964a8b",
    "QCD": "#9c9ca1",
}

## 1. Load data and build class labels

Class labels come directly from the `fj_isWplus`/`fj_isWminus`/`fj_isZ`/`fj_isQCD_Matched` flags
(confirmed mutually exclusive by construction), not from filenames — so this works regardless of
how files are named or merged.

Note: these columns come back from parquet as `float64` (0./1.), not `bool` — cast explicitly.

In [ ]:
files = sorted(glob.glob(f"{PARQUET_DIR}/*.parquet"))
print(f"Found {len(files)} files:")
for f in files:
    print(" ", f)

data = ak.concatenate([ak.from_parquet(f) for f in files])
print(f"\nTotal rows: {len(data)}")

is_wplus = ak.to_numpy(data["fj_isWplus"]).astype(bool)
is_wminus = ak.to_numpy(data["fj_isWminus"]).astype(bool)
is_z = ak.to_numpy(data["fj_isZ"]).astype(bool)
is_qcd = ak.to_numpy(data["fj_isQCD_Matched"]).astype(bool)

assert (is_wplus.astype(int) + is_wminus.astype(int) + is_z.astype(int) + is_qcd.astype(int) <= 1).all(), \
    "class flags are not mutually exclusive -- investigate before trusting downstream plots"

class_label = np.full(len(data), "Unknown", dtype=object)
class_label[is_wplus] = "Wplus"
class_label[is_wminus] = "Wminus"
class_label[is_z] = "Z"
class_label[is_qcd] = "QCD"

masks = {c: (class_label == c) for c in CLASS_ORDER}
print("Class yields:", {c: int(masks[c].sum()) for c in CLASS_ORDER})

## 2. `fj_msoftdrop` mass peak — the headline validation plot

W/Z should show a clear resonance peak (~80/91 GeV); QCD should fall off smoothly with no peak.
If this doesn't look right, nothing downstream is worth trusting yet.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
bins = np.linspace(0, 250, 60)
for c in CLASS_ORDER:
    vals = ak.to_numpy(data["fj_msoftdrop"][masks[c]])
    ax.hist(vals, bins=bins, histtype="step", density=True, label=c, color=CLASS_COLORS[c], linewidth=2)
ax.set_xlabel("fj_msoftdrop [GeV]")
ax.set_ylabel("Normalized")
ax.legend()
fig.tight_layout()

## 3. Jet kinematics per class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 7))
pt_bins = np.linspace(200, 1500, 40)
eta_bins = np.linspace(-2.5, 2.5, 40)
for c in CLASS_ORDER:
    axes[0].hist(ak.to_numpy(data["fj_pt"][masks[c]]), bins=pt_bins, histtype="step", density=True, label=c, color=CLASS_COLORS[c], linewidth=2)
    axes[1].hist(ak.to_numpy(data["fj_eta"][masks[c]]), bins=eta_bins, histtype="step", density=True, label=c, color=CLASS_COLORS[c], linewidth=2)
axes[0].set_xlabel("fj_pt [GeV]"); axes[0].legend()
axes[1].set_xlabel("fj_eta"); axes[1].legend()
fig.tight_layout()

## 4. Decay-mode / flavor composition

Sanity check against known SM branching ratios (Z: bb ~21.5%, cc ~17%, light ~61%),
and see how much of the QCD background is heavy-flavor (the hardest background for Z->bb/cc).

In [ ]:
def frac(mask_num, mask_den):
    n = int(ak.sum(mask_den))
    return float(ak.sum(mask_num)) / n if n else np.nan

w_submodes = {
    "W+ ud": frac(data["fj_isWplus_ud"], data["fj_isWplus"]),
    "W+ cs": frac(data["fj_isWplus_cs"], data["fj_isWplus"]),
    "W- ud": frac(data["fj_isWminus_ud"], data["fj_isWminus"]),
    "W- cs": frac(data["fj_isWminus_cs"], data["fj_isWminus"]),
}
z_submodes = {
    "Z bb": frac(data["fj_isZ_bb"], data["fj_isZ"]),
    "Z cc": frac(data["fj_isZ_cc"], data["fj_isZ"]),
    "Z qq": frac(data["fj_isZ_qq"], data["fj_isZ"]),
}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(list(w_submodes.keys()), list(w_submodes.values()), color=CLASS_COLORS["Wplus"])
axes[0].set_ylabel("Fraction")
axes[1].bar(list(z_submodes.keys()), list(z_submodes.values()), color=CLASS_COLORS["Z"])
fig.tight_layout()

In [ ]:
qcd_flavors = {
    "bb": frac(data["fj_isQCD_bb"], data["fj_isQCD_Matched"]),
    "b": frac(data["fj_isQCD_b"], data["fj_isQCD_Matched"]),
    "cc": frac(data["fj_isQCD_cc"], data["fj_isQCD_Matched"]),
    "c": frac(data["fj_isQCD_c"], data["fj_isQCD_Matched"]),
    "other": frac(data["fj_isQCD_other"], data["fj_isQCD_Matched"]),
}
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(list(qcd_flavors.keys()), list(qcd_flavors.values()), color=CLASS_COLORS["QCD"])
ax.set_ylabel("Fraction of QCD jets")
fig.tight_layout()

## 5. PF-candidate feature sanity checks

`pfcands_d0sig`/`pfcands_dzsig` split by charged vs. neutral candidates — neutral (untracked)
candidates should now pile up at exactly 0 (the sentinel-division fix), not at a fake ~1.0.

In [ ]:
has_track = ak.to_numpy(ak.flatten(data["pfcands_charge"] != 0, axis=None))
d0sig_flat = ak.to_numpy(ak.flatten(data["pfcands_d0sig"], axis=None))
dzsig_flat = ak.to_numpy(ak.flatten(data["pfcands_dzsig"], axis=None))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
bins = np.linspace(-5, 5, 60)
axes[0].hist(d0sig_flat[has_track], bins=bins, histtype="step", density=True, label="charged", color="#5790fc", linewidth=2)
axes[0].hist(d0sig_flat[~has_track], bins=bins, histtype="step", density=True, label="neutral", color="#e42536", linewidth=2)
axes[0].set_xlabel("pfcands_d0sig"); axes[0].legend()
axes[1].hist(dzsig_flat[has_track], bins=bins, histtype="step", density=True, label="charged", color="#5790fc", linewidth=2)
axes[1].hist(dzsig_flat[~has_track], bins=bins, histtype="step", density=True, label="neutral", color="#e42536", linewidth=2)
axes[1].set_xlabel("pfcands_dzsig"); axes[1].legend()
fig.tight_layout()

print("neutral d0sig all zero:", bool(np.all(d0sig_flat[~has_track] == 0)))

## 6. PF-candidate radial profile and multiplicity per class

W/Z jets should show a collimated, two-pronged structure; QCD a broader single-core profile
with (typically) higher candidate multiplicity.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
dr_bins = np.linspace(0, 0.8, 40)
ptrel_bins = np.linspace(0, 1, 40)
for c in CLASS_ORDER:
    dr_vals = ak.to_numpy(ak.flatten(data["pfcands_dr"][masks[c]], axis=None))
    ptrel_vals = ak.to_numpy(ak.flatten(data["pfcands_ptrel"][masks[c]], axis=None))
    axes[0].hist(dr_vals, bins=dr_bins, histtype="step", density=True, label=c, color=CLASS_COLORS[c], linewidth=2)
    axes[1].hist(ptrel_vals, bins=ptrel_bins, histtype="step", density=True, label=c, color=CLASS_COLORS[c], linewidth=2)
axes[0].set_xlabel("pfcands_dr"); axes[0].legend()
axes[1].set_xlabel("pfcands_ptrel"); axes[1].legend()
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
mult_bins = np.arange(0, 150, 5)
for c in CLASS_ORDER:
    mult = ak.to_numpy(ak.num(data["pfcands_dr"][masks[c]], axis=1))
    ax.hist(mult, bins=mult_bins, histtype="step", density=True, label=c, color=CLASS_COLORS[c], linewidth=2)
ax.set_xlabel("PF candidate multiplicity")
ax.legend()
fig.tight_layout()

## 7. Secondary vertices by QCD flavor label

Checks whether the displaced-vertex information is actually discriminating for heavy-flavor QCD
(the background that matters most for Z->bb/cc).

In [ ]:
qcd_flavor_masks = {
    "bb": ak.to_numpy(data["fj_isQCD_bb"]).astype(bool),
    "cc": ak.to_numpy(data["fj_isQCD_cc"]).astype(bool),
    "other": ak.to_numpy(data["fj_isQCD_other"]).astype(bool),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sv_mult_bins = np.arange(0, 8, 1)
dxysig_bins = np.linspace(0, 20, 40)
for label, m in qcd_flavor_masks.items():
    n_sv = ak.to_numpy(ak.num(data["sv_dxysig"][m], axis=1))
    axes[0].hist(n_sv, bins=sv_mult_bins, histtype="step", density=True, label=label, linewidth=2)
    dxysig_vals = ak.to_numpy(ak.flatten(data["sv_dxysig"][m], axis=None))
    if len(dxysig_vals):
        axes[1].hist(dxysig_vals, bins=dxysig_bins, histtype="step", density=True, label=label, linewidth=2)
axes[0].set_xlabel("N secondary vertices"); axes[0].legend()
axes[1].set_xlabel("sv_dxysig"); axes[1].legend()
fig.tight_layout()